## XGBM & LGBM
```
Objective:
The objective of this assignment is to compare the performance of Light GBM and XG Boost algorithms using the Titanic dataset. 

Exploratory Data Analysis (EDA):
1.	Load the Titanic dataset using Python's pandas library.
2.	Check for missing values.
3.	Explore data distributions using histograms and box plots.
4.	Visualize relationships between features and survival using scatter plots and bar plots.

Data Preprocessing:
1.	Impute missing values.
2.	Encode categorical variables using one-hot encoding or label encoding. 
3.	If needed you can apply more preprocessing methods on the given dataset.

Building Predictive Models:
1.	Split the preprocessed dataset into training and testing sets.
2.	Choose appropriate evaluation metrics (e.g., accuracy, precision, recall, F1-score) for model evaluation.
3.	Build predictive models using LightGBM and XGBoost algorithms.
4.	Train the models on the training set and evaluate their performance on the testing set.
5.	Use techniques like cross-validation and hyperparameter tuning to optimize model performance.

Comparative Analysis:
1.	Compare the performance metrics (e.g., accuracy, precision, recall) of LightGBM and XGBoost models.
2.	Visualize and interpret the results to identify the strengths and weaknesses of each algorithm.

Submission Requirements:
Well-commented code explaining each step of the analysis.
Visualizations with appropriate titles and labels.
A brief report summarizing the comparative analysis results and practical implications.


```

In [ ]:
# Understanding data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Loading dataset to the envoroment
dataset = pd.read_csv("diabetes.csv")

# Working on a cloned copy
df = dataset.copy()

print("\n<----------INFO----------->\n")
print(df.info())

print("\n<-----------DESCRIBE ONLY NUMERICAL---------->\n")
print(df.describe())

print("\n<---------DESCRIBE ALL NUMERICAL AND CATEGORICAL--------->\n")
print(df.describe(include='all'))

print("\n<---------MISSING VALUES--------->\n")
print(df.isnull().sum())

## Exploratory Data Analysis(EDA):

In [ ]:
# Extracting Coiumns 
target = 'Outcome'
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.remove('Outcome')

#Checking the uniqueness of the target column
print(df[target].unique())

In [ ]:
# Univariate Analysis
for col in numerical_cols:
    plt.figure(figsize=(18,9))

    plt.subplot(1,2,1)
    plt.title(f"Histogram for {col}")
    sns.histplot(df[col],kde=True)

    plt.subplot(1,2,2)
    plt.title(f"Boxplot for {col}")
    sns.boxplot(df[col])

    plt.show()

In [ ]:
# Bivariate Analysis

# Numerical features vs target
for col in numerical_cols:
    plt.figure(figsize=(18,9))
    plt.title(f"Boxplot between {col} and {target}")
    sns.boxplot(x=target,y=col, data=df)

    plt.show()

In [ ]:
# Bivarate analysis between the numerical variables
for i,x_col in enumerate(numerical_cols):
    for y_col in numerical_cols[i+1:]:
        plt.figure(figsize=(18,9))
        sns.scatterplot(x=x_col, y= y_col,data=df)
        plt.title(f"Scatterplot between the {x_col} and {y_col}")

        plt.show()

In [ ]:
# Considering the output treating the features accordingly
# Treating unrealistic values like 0 in case of insulin or other features.
for col in numerical_cols:
    for cls in [0, 1]:  
        median_class = df.loc[df['Outcome'] == cls, col].median()
        
        # IQR for this class
        Q1 = df.loc[df['Outcome'] == cls, col].quantile(0.25)
        Q3 = df.loc[df['Outcome'] == cls, col].quantile(0.75)
        IQR = Q3 - Q1
        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR
        
        # Replace outliers for this class with class-specific median
        df.loc[(df['Outcome'] == cls) & 
               ((df[col] < lower_limit) | (df[col] > upper_limit)), col] = median_class


In [ ]:
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers")


In [ ]:
for col in numerical_cols:
    plt.figure(figsize=(18,9))
    plt.title(f"Boxplot between {col} and {target}")
    sns.boxplot(x=target,y=col, data=df)

    plt.show()

## Data Partition And Model Building

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Dividing dataset- target and independent
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

# Train-Test split
X_train, X_test,y_train,y_test = train_test_split(X,y, test_size=0.30,random_state=42,stratify=y)

xgb_model = XGBClassifier(eval_metric="logloss",random_state=42)

parameters ={
    'n_estimators':[50,100],
    'max_depth':[3,5],
    'learning_rate':[0.05,0.1],
    'subsample':[0.8,1.0],
    'min_child_weight':[1,3],
    'gamma': [0, 0.25]
}

xgb_grid_model = GridSearchCV(xgb_model,parameters,scoring='f1', cv=5)

xgb_grid_model.fit(X_train, y_train)

In [ ]:
# Best model evaluation
print("Best params:", xgb_grid_model.best_params_)
print("Best CV score:", xgb_grid_model.best_score_)

best_xgb = xgb_grid_model.best_estimator_
y_pred = best_xgb.predict(X_test)

print("\n<-------Classification Report---------->\n")
print(classification_report(y_pred, y_test))

print("\n<--------Best Test Score--------->\n")
print(accuracy_score(y_pred,y_test))


In [ ]:
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV

lgb_model = lgb.LGBMClassifier(random_state=42,verbose=-1)
params_lgb = {
    'n_estimators':[50,100,200],
    'max_depth':[3,5,7],
    'learning_rate':[0.05,0.1],
    'subsample':[0.8,1.0],
    'num_leaves':[15,31,63],
    'min_child_samples':[1,5,10]
}
lgb_rand_model = RandomizedSearchCV(lgb_model, params_lgb, n_iter=20,cv=5,scoring = 'f1',random_state=42)
lgb_rand_model.fit(X_train,y_train)


In [ ]:
best_lgb = lgb_rand_model.best_estimator_
y_pred_lgb = best_lgb.predict(X_test)

print("\n<---------Results form the Lightgbm--------->\n")
print("Best parameters:", lgb_rand_model.best_params_)
print("Test Accuracy:",accuracy_score(y_pred_lgb,y_test))
print("\n<---------Classification report is given by----------->\n")
print(classification_report(y_pred_lgb,y_test))


## Comparative Analysis (XGBM vs LGBM)
```
1.Accuracy
XGBM: 78.8%
LGBM: 80.1% → Better

2.Majority Class (0 = <=50K)
XGBM: Precision 0.89, Recall 0.80, F1 = 0.85
LGBM: Precision 0.91, Recall 0.81, F1 = 0.86 → Better

3.Minority Class (1 = >50K)
XGBM: Precision 0.59, Recall 0.75, F1 = 0.66
LGBM: Precision 0.60, Recall 0.78, F1 = 0.68 → Better

4.Macro Average F1
XGBM: 0.75
LGBM: 0.77 → Better
```